# 01 — DM, SX, MX, and matrix semantics

**Objectives.** Choose the right CasADi type; construct and slice matrices; move data to and from NumPy.

`DM` stores numerical sparse matrices. `SX` stores scalar expression trees and excels at elementwise formulas. `MX` stores matrix-operation graphs and is usually preferable for large composed algorithms and `Opti`. All three preserve two-dimensional matrix shapes.


## Learning path and assumptions

**Prerequisites:** Chapter 00 and matrix multiplication.

**Problem:** Choose numerical DM or symbolic SX/MX without creating NumPy object arrays.

**Assumptions and conventions:** All quantities in this lesson are dimensionless.

**How to read the results:** Check shapes before values: a valid expression with the wrong shape can encode the wrong mathematical problem.

### Resolve the local checkout

This setup locates repository-local presentation and numerical helpers and enables autoreload. It changes Python import resolution, not the mathematical model. Run it once after restarting the kernel.

In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Compare numeric, scalar-symbolic, and graph-symbolic matrices

This cell introduces the three core CasADi matrix families. `DM` stores
known numerical values, `SX.sym` creates scalar expression graphs, and
`MX.sym` creates matrix-valued computational graphs. Both symbolic types
support the same matrix algebra, but their internal graph representations
favor different workloads. `vertcat` and `horzcat` demonstrate explicit
row-wise and column-wise concatenation.

For

$$
A=\begin{bmatrix}1&2\\3&4\end{bmatrix},\qquad
x=\begin{bmatrix}x_0\\x_1\end{bmatrix},
$$

the symbolic matrix product constructed by `@` is

$$
Ax=\begin{bmatrix}x_0+2x_1\\3x_0+4x_1\end{bmatrix}.
$$

Concatenation changes only structure:
$\operatorname{vertcat}(x,x)\in\mathbb{R}^{4\times1}$ and
$\operatorname{horzcat}(x,x)\in\mathbb{R}^{2\times2}$.


In [2]:
import casadi as ca
import numpy as np

# DM holds known numerical data; SX and MX represent symbolic expressions.
coefficient_matrix = ca.DM([[1, 2], [3, 4]])
sx_vector = ca.SX.sym("sx_vector", 2)
mx_vector = ca.MX.sym("mx_vector", 2)

# The @ operator creates symbolic matrix products for either graph type.
print("DM @ SX:", coefficient_matrix @ sx_vector)
print("DM @ MX:", coefficient_matrix @ mx_vector)

# vertcat stacks rows, while horzcat stacks columns.
print(
    "vertical shape:",
    ca.vertcat(sx_vector, sx_vector).shape,
    "horizontal shape:",
    ca.horzcat(sx_vector, sx_vector).shape,
)

# full() materializes a DM as a dense NumPy-compatible array.
round_trip = np.asarray(coefficient_matrix.full())
assert round_trip.shape == (2, 2)

DM @ SX: [(sx_vector_0+(2*sx_vector_1)), ((3*sx_vector_0)+(4*sx_vector_1))]
DM @ MX: mac(
[[1, 2], 
 [3, 4]],mx_vector,zeros(2x1))
vertical shape: (4, 1) horizontal shape: (2, 2)


## Indexing and a trap

CasADi indexing is zero-based in Python. `*` is elementwise multiplication and `@`/`ca.mtimes` is matrix multiplication. NumPy arrays containing symbolic objects become slow object arrays; put CasADi on the outside of mixed operations.


### Package an affine expression as a reusable CasADi function

This cell first shows that indexing a CasADi vector returns a
$1\times1$ symbolic matrix. It then uses `ca.Function` to turn an SX
expression into a callable graph with named input and output interfaces.
A CasADi function can be called with ordinary lists or NumPy values; those
values are converted to `DM` before the symbolic graph is evaluated.

The implemented affine map is

$$
y = Ax + \mathbf{1}
  = \begin{bmatrix}1&2\\3&4\end{bmatrix}x
    + \begin{bmatrix}1\\1\end{bmatrix}.
$$

At $x=[1,1]^\mathsf{T}$, the expected result is
$y=[4,8]^\mathsf{T}$. The NumPy assertion checks this numerical value
after evaluating the CasADi graph.


In [3]:
# CasADi indexing returns a 1x1 symbolic matrix rather than a Python scalar.
stacked_vector = ca.vertcat(sx_vector, 2 * sx_vector)
assert stacked_vector[1].shape == (1, 1)

# Function packages a symbolic graph behind named numerical inputs and outputs.
affine_map = ca.Function(
    "affine",
    [sx_vector],
    [coefficient_matrix @ sx_vector + ca.DM.ones(2, 1)],
    ["x"],
    ["y"],
)
result = affine_map([1, 1])
np.testing.assert_allclose(result.full().ravel(), [4, 8])
print("affine([1, 1]) =", result.full().ravel())

affine([1, 1]) = [4. 8.]


**Recap.** Use `DM` for CasADi numeric I/O, `SX` for scalar-style graphs, and `MX` for matrix/composed graphs. Exercise: build a block matrix without NumPy object arrays. [Exercises](../exercises/01_foundations.ipynb).


## Experiment: shape and graph family

For $A\in\mathbb R^{m\times n}$ and $x\in\mathbb R^{n\times1}$, $Ax$ has $m$ rows. This experiment constructs a new DM, SX, or MX object and evaluates the same product. The assertion checks a mathematical invariant, independently of graph representation.

In [4]:
def shape_experiment(rows=3, columns=2, family="SX"):
    matrix = ca.DM(np.arange(rows * columns).reshape(rows, columns) + 1)
    if family == "DM":
        vector = ca.DM.ones(columns)
        result = matrix @ vector
    else:
        vector = getattr(ca, family).sym("shape_x", columns)
        function = ca.Function("shape_product", [vector], [matrix @ vector])
        result = function(np.ones(columns))
    assert result.shape == (rows, 1)
    print(f"{family}: A {matrix.shape} @ x {(columns, 1)} -> {result.shape}")
    print(result)
shape_experiment()

SX: A (3, 2) @ x (2, 1) -> (3, 1)
[3, 7, 11]


**Predict:** does changing SX to MX change the numerical answer or only its graph representation?

### Run your own experiment

The default above is already computed. Change the controls, predict the result, and press **Run experiment**. **Reset controls** restores the initial settings. Solver calls are explicit; dragging controls does not launch optimizations. Python controls require a live JupyterLab kernel.

In [5]:
from casadi_tutorial.interactive import experiment
try:
    import ipywidgets as widgets
except ImportError:
    print("Install the course environment to enable live controls.")
else:
    experiment(shape_experiment, dict(rows=widgets.IntSlider(value=3,min=1,max=6), columns=widgets.IntSlider(value=2,min=1,max=6), family=widgets.Dropdown(options=["DM","SX","MX"],value="SX")), title='Matrix types and dimensions')

## Recap and independent practice

State the mathematical object constructed in this module, the numerical assumptions it requires, and one check that could reveal a wrong result. Change one experimental setting at a time and explain the observed change before continuing.